In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

import pandas as pd


In [ ]:
input_file = DATA_ROOT / "interdependence_paper_supplementary_data.xlsx"
output_file = DATA_ROOT / "interdependence_paper_supplementary_data_new.xlsx" 
    

In [ ]:
#open the old excel sheets
#df_sources = pd.read_excel(input_file, sheet_name='paper_source_data')
df_std_sources = pd.read_excel(input_file, sheet_name='standardized_sources')
df_vis = pd.read_excel(input_file, sheet_name='visualization_names')


In [ ]:
df_overwrite = pd.read_excel(input_file, sheet_name='unique_institutions')


def clean_text_value(value):
    # Normalize missing values and incidental whitespace without changing wording.
    if pd.isna(value):
        return ""
    text = str(value).replace("\xa0", " ").strip()
    if text.casefold() in {"nan", "<na>", "none"}:
        return ""
    return " ".join(text.split())


def split_semicolon_values(value):
    # Split, clean, and deduplicate list values case-insensitively while preserving order.
    values = []
    seen = set()
    for item in clean_text_value(value).split(";"):
        cleaned = clean_text_value(item)
        key = cleaned.casefold()
        if cleaned and key not in seen:
            seen.add(key)
            values.append(cleaned)
    return values


def normalize_semicolon_values(value):
    return "; ".join(split_semicolon_values(value))


def map_semicolon_values(value, lookup, field_label):
    # Require every holder to have reviewed metadata, then deduplicate mapped results.
    mapped_values = []
    seen = set()
    for item in split_semicolon_values(value):
        if item not in lookup:
            raise KeyError(f"No {field_label} mapping found for data holder: {item!r}")
        mapped = clean_text_value(lookup[item])
        key = mapped.casefold()
        if mapped and key not in seen:
            seen.add(key)
            mapped_values.append(mapped)
    return "; ".join(mapped_values)


# Clean the reviewed mapping table before constructing lookup dictionaries.
for column in df_overwrite.select_dtypes(include="object").columns:
    df_overwrite[column] = df_overwrite[column].map(clean_text_value)

# Normalize the name/list fields used by this notebook before matching.
# for column in ["standardized_data_resource_names", "standardized_data_holder_names"]:
#     if column in df_sources.columns:
#         df_sources[column] = df_sources[column].map(normalize_semicolon_values)

for column in ["raw_name", "standardized_data_resource_name"]:
    if column in df_std_sources.columns:
        df_std_sources[column] = df_std_sources[column].map(clean_text_value)
for column in ["standardized_data_holder_name", "standardized_data_holder_type", "standardized_data_holder_country_ISO2", "URL"]:
    if column in df_std_sources.columns:
        df_std_sources[column] = df_std_sources[column].map(normalize_semicolon_values)

for column in ["standardized_data_resource_name", "data_resource_for_visualization"]:
    if column in df_vis.columns:
        df_vis[column] = df_vis[column].map(clean_text_value)
for column in ["standardized_data_holder_name", "data_holder_for_visualization", "standardized_data_holder_type", "standardized_data_holder_country_ISO2"]:
    if column in df_vis.columns:
        df_vis[column] = df_vis[column].map(normalize_semicolon_values)

In [ ]:
df_overwrite.head()

### 1. Data holders

In [ ]:
#merge duplicate data holders manually
overwrite_dict_name = dict(zip(list(df_overwrite['data_holder'].str.replace('\xa0', ' ')), list(df_overwrite['data_holder'].replace({"UNESCO-IOC":"IOC-UNESCO","European Environment Agency":"European Environment Agency (EEA)", "EMBL-EBI": "EMBL", "USDA NASS":"USDA"
}))))

In [ ]:
# Apply reviewed holder-name mappings to every sheet.
df_std_sources['standardized_data_holder_name'] = df_std_sources['standardized_data_holder_name'].map(
    lambda value: map_semicolon_values(value, overwrite_dict_name, "name")
)
df_vis['standardized_data_holder_name'] = df_vis['standardized_data_holder_name'].map(
    lambda value: map_semicolon_values(value, overwrite_dict_name, "name")
)
# df_sources['standardized_data_holder_names'] = df_sources['standardized_data_holder_names'].map(
#     lambda value: map_semicolon_values(value, overwrite_dict_name, "name")
# )



### 2. Data holder type

In [ ]:
list(set(df_overwrite['holder_type_new']))

In [ ]:
overwrite_dict_type = dict(zip(list(df_overwrite['data_holder'].str.replace('\xa0', ' ')), list(df_overwrite['holder_type_new'])))

In [ ]:
# Apply reviewed holder-type mappings.
df_std_sources['standardized_data_holder_type'] = df_std_sources['standardized_data_holder_name'].map(
    lambda value: map_semicolon_values(value, overwrite_dict_type, "type")
)
df_std_sources.head()

In [ ]:
# Apply the same reviewed holder-type mappings to the visualization sheet.
df_vis['standardized_data_holder_type'] = df_vis['standardized_data_holder_name'].map(
    lambda value: map_semicolon_values(value, overwrite_dict_type, "type")
)
df_vis.head()

In [ ]:
#replace multiple types with 'hybrid'
holder_types_disamb = []
for holder_type in list(df_std_sources['standardized_data_holder_type']): 
    if len(set(holder_type.split(';'))) >= 2:
        holder_types_disamb.append('hybrid')
    else: 
        holder_types_disamb.append(holder_type)

df_std_sources['standardized_data_holder_type'] = holder_types_disamb 



holder_types_disamb = []
for holder_type in list(df_vis['standardized_data_holder_type']): 
    if len(set(holder_type.split(';'))) >= 2:
        holder_types_disamb.append('hybrid')
    else: 
        holder_types_disamb.append(holder_type)

df_vis['standardized_data_holder_type'] = holder_types_disamb 

### 3. Countries

In [ ]:
# EU membership takes precedence over any manually entered national country code.
eu_holder_mask = pd.to_numeric(df_overwrite['EU'], errors='coerce').eq(1)
df_overwrite.loc[eu_holder_mask, 'holder_ISO2_country_new'] = 'EU'

# Build the reviewed holder-country lookup after applying the EU override.
overwrite_dict_country = dict(zip(df_overwrite['data_holder'], df_overwrite['holder_ISO2_country_new']))
print(f"Set country to EU for {eu_holder_mask.sum():,} reviewed data holders.")

In [ ]:
overwrite_dict_country

In [ ]:
# Apply reviewed holder-country mappings to both metadata sheets.
df_std_sources['standardized_data_holder_country_ISO2'] = df_std_sources['standardized_data_holder_name'].map(
    lambda value: map_semicolon_values(value, overwrite_dict_country, "country")
)
df_vis['standardized_data_holder_country_ISO2'] = df_vis['standardized_data_holder_name'].map(
    lambda value: map_semicolon_values(value, overwrite_dict_country, "country")
)

df_vis.head()


In [ ]:
#replace multiple countries with 'transnational/ambiguous'
holder_countries_disamb = []
for holder_country in list(df_std_sources['standardized_data_holder_country_ISO2']): 
    if len(set(holder_country.split(';'))) >= 2:
        holder_countries_disamb.append('transnational/ambiguous')
    else: 
        holder_countries_disamb.append(holder_country)

df_std_sources['standardized_data_holder_country_ISO2'] = holder_countries_disamb 



holder_countries_disamb = []
for holder_country in list(df_vis['standardized_data_holder_country_ISO2']): 
    if len(set(holder_country.split(';'))) >= 2:
        holder_countries_disamb.append('transnational/ambiguous')
    else: 
        holder_countries_disamb.append(holder_country)

df_vis['standardized_data_holder_country_ISO2'] = holder_countries_disamb 

### 4. Overwrite old sheets

In [ ]:
import shutil
import pandas as pd

# Copy original workbook
shutil.copy(input_file, output_file)

# Normalize the final exported list fields after all reviewed mappings.
# for column in ["standardized_data_resource_names", "standardized_data_holder_names"]:
#     if column in df_sources.columns:
#         df_sources[column] = df_sources[column].map(normalize_semicolon_values)

for column in ["standardized_data_holder_name", "standardized_data_holder_type", "standardized_data_holder_country_ISO2", "URL"]:
    if column in df_std_sources.columns:
        df_std_sources[column] = df_std_sources[column].map(normalize_semicolon_values)

for column in ["standardized_data_holder_name", "data_holder_for_visualization", "standardized_data_holder_type", "standardized_data_holder_country_ISO2"]:
    if column in df_vis.columns:
        df_vis[column] = df_vis[column].map(normalize_semicolon_values)
for column in ["standardized_data_resource_name", "data_resource_for_visualization"]:
    if column in df_vis.columns:
        df_vis[column] = df_vis[column].map(clean_text_value)

updated_sheets = {
    # "paper_source_data": df_sources,
    "standardized_sources": df_std_sources,
    "visualization_names": df_vis
}

# Replace updated sheets
with pd.ExcelWriter(
    output_file,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    for sheet_name, df in updated_sheets.items():
        df.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )